# GQA + QK-Norm + RoPE Attention

源码导航：[`repeat_kv`](../../../core/attention/walkie_attention.py#L23)、[`WalkieCausalSelfAttention`](../../../core/attention/walkie_attention.py#L34)、[`sdpa` 分支](../../../core/attention/walkie_attention.py#L116)、[`eager` 分支](../../../core/attention/walkie_attention.py#L123)。

标准 causal self-attention 为：

$$
\operatorname{Attn}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^\top}{\sqrt{d_h}}+M_{causal}\right)V
$$

Walkie 的注意力在这个基础上做三处现代化改进：GQA 减少 K/V 头以节省 KV cache 与参数，QK-Norm 控制每个 head 的 Q/K 尺度，RoPE 在 Q/K 上注入相对位置信息。工程路径优先使用 PyTorch SDPA，同时保留 eager 分支用于教学和对齐。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.attention.walkie_attention import WalkieCausalSelfAttention, repeat_kv

## 1. GQA 的 head 展开

In [ ]:
kv = torch.randn(2, 2, 8, 16)  # B, H_kv, T, D
expanded = repeat_kv(kv, n_rep=4)
print('kv      :', tuple(kv.shape))
print('expanded:', tuple(expanded.shape))
print('head 0 == head 1:', torch.allclose(expanded[:, 0], expanded[:, 1]))

## 2. 前向与投影尺寸

In [ ]:
attn = WalkieCausalSelfAttention(
    n_embd=128, n_head=8, n_head_kv=2, head_dim=16, max_seq_len=64,
    dropout=0.0, qk_norm=True, attn_impl='sdpa',
)
x = torch.randn(2, 12, 128)
y = attn(x)
print('output:', tuple(y.shape))
for name in ['q_proj', 'k_proj', 'v_proj', 'o_proj']:
    mod = getattr(attn, name)
    print(f'{name:6s}', tuple(mod.weight.shape))

---

## 延伸阅读与参考资料

### 核心论文
- **Attention Is All You Need**: Vaswani et al., 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
- **GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints**: Ainslie et al., 2023. [arXiv:2305.13245](https://arxiv.org/abs/2305.13245)
- **FlashAttention**: Dao et al., 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)

### 工程实现
- **PyTorch SDPA**: [docs](https://pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)
- **Hugging Face LlamaAttention / GQA**: [source](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)